# VeRi Canon Fusion Eval

Canonical, fully self-contained **inference-only** reproduction of the deployed VeRi-776
two-stream fusion headline (**~93.30% mAP / 98.45% R1**).

Loads two FROZEN checkpoints from `gumfreddy/veri776-canonical-weights` (no training, no repo clone):
- Stream 1: `transreid_a5alpha_veri776.pth` -> TransReID ViT-B/16 CLIP, concat_patch_flip 1536-D (flip-TTA, img 224, CLIP norm).
- Stream 2: `clipsenet_v7_veri776.pth` -> CLIP-SENet (ResNet101-IBN-a + TinyCLIP-45M), 2048-D (img 320, ImageNet norm, no TTA).

**Headline fusion (deployed):** AQE k=3 BOTH streams -> sim = `0.3*S_transreid + 0.7*S_clipsenet`
-> k-reciprocal rerank (k1=80, k2=15, lambda=0.2) -> Market-1501 protocol (same-(pid AND camid) junk filter).

Also emits per-stream standalone rows, the full score-fusion w-sweep, and the concat-feature alpha grid
for the paper tables. Logs SHA-256 of both checkpoints and asserts backbone provenance (fail hard on fallback).

In [ ]:
# === Setup + pinned deps (inference only) ===
import os
import sys
import subprocess
from pathlib import Path

print("Python:", sys.version)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# torch pinned to cu124 so the kernel runs on EITHER Tesla T4 (sm_75) or P100 (sm_60);
# Kaggle stock torch 2.10+cu128 dropped sm_60 and crashes on P100.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "torch==2.4.1+cu124", "torchvision==0.19.1+cu124",
        "--index-url", "https://download.pytorch.org/whl/cu124",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "timm==1.0.11",
        "open_clip_torch==2.30.0",
        "pretrainedmodels==0.7.4",
    ],
    check=True,
)

import json
import re
import time
import gc
import hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
print("torchvision:", torchvision.__version__)
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# Reproducibility (inference is deterministic, but pin anyway)
SEED = 0
import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

WORKING = Path("/kaggle/working")
CANON_WEIGHTS_DIR = Path("/kaggle/input/paper-a5weights")  # primary; per-file rglob discovery below
STREAM1_CKPT_NAME = "vehicle_transreid_vit_base_veri776.pth"  # ALT S1 candidate (mrkdagods/mtmc-weights, deployed serving)
STREAM2_CKPT_NAME = "clipsenet_v8_veri776.pth"  # stronger S2 (v6 recipe @320/P8 + 40ep)


## Inlined model: TransReID (verbatim from `src/stage2_features/transreid_model.py`, no `src.` import, no repo clone)

In [ ]:
"""TransReID model for inference in the MTMC pipeline.

TransReID (He et al., ICCV 2021) with:
- ViT backbone (via timm) — supports CLIP ViT-Base and standard ViTs
- Side Information Embedding (SIE) — camera-aware tokens broadcast to ALL tokens
- Jigsaw Patch Module (JPM) — used only during training
- BNNeck + optional projection for deployment features
- norm_pre support for CLIP ViT compatibility (critical for CLIP backbones)

This module is used by Stage 2 (feature extraction) to load
TransReID weights trained on Kaggle (Notebook 07/08).

Architecture must match the training notebook (NB08) exactly:
- Weight key names: ``bn`` (not ``bn_global``), ``cls_head`` (not ``classifier``)
- SIE broadcasts to ALL tokens (not just CLS)
- norm_pre called before transformer blocks (critical for CLIP ViTs)
- Identity projection when embed_dim == vit_dim
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
import logging
logger = logging.getLogger("transreid")


class TransReID(nn.Module):
    """TransReID: ViT + SIE + JPM for re-identification.

    During inference, returns L2-normalized ``embed_dim``-dimensional features.
    During training, returns (cls_score, proj_feat[, jpm_score]).

    v6 CRITICAL FIX: Includes timm's norm_pre for CLIP compatibility.
    CLIP ViTs use pre-LayerNorm that standard ViTs lack — skipping it
    completely destroys pretrained features.
    """

    def __init__(
        self,
        num_classes: int = 1,
        num_cameras: int = 0,
        embed_dim: int = 768,
        vit_model: str = "vit_base_patch16_clip_224.openai",
        pretrained: bool = False,
        sie_camera: bool = True,
        jpm: bool = True,
        img_size: tuple[int, int] | None = None,
    ):
        super().__init__()
        import timm

        self.sie_camera = sie_camera and num_cameras > 0
        self.jpm = jpm

        # ViT backbone — pass img_size when it differs from the timm default
        # (e.g. person ReID uses 256×128 → grid 16×8 = 128 patches)
        timm_kwargs = dict(pretrained=pretrained, num_classes=0)
        if img_size is not None:
            timm_kwargs["img_size"] = img_size
        self.vit = timm.create_model(vit_model, **timm_kwargs)
        self.vit_dim = self.vit.embed_dim  # 768 for ViT-Base, 384 for ViT-Small
        self.num_blocks = len(self.vit.blocks)

        # Detect architecture features
        has_norm_pre = hasattr(self.vit, "norm_pre") and not isinstance(
            self.vit.norm_pre, nn.Identity
        )
        logger.debug(
            f"TransReID: {vit_model}, vit_dim={self.vit_dim}, "
            f"norm_pre={type(self.vit.norm_pre).__name__} (active={has_norm_pre}), "
            f"blocks={self.num_blocks}"
        )

        # SIE: camera embedding broadcast to ALL tokens (per TransReID paper)
        if self.sie_camera:
            self.sie_embed = nn.Parameter(
                torch.zeros(num_cameras, 1, self.vit_dim)
            )
            nn.init.trunc_normal_(self.sie_embed, std=0.02)

        # BNNeck (named 'bn' to match NB08 training checkpoint keys)
        self.bn = nn.BatchNorm1d(self.vit_dim)
        self.bn.bias.requires_grad_(False)

        # Projection: Identity when embed_dim == vit_dim (e.g., 768 → 768)
        self.proj = (
            nn.Linear(self.vit_dim, embed_dim, bias=False)
            if embed_dim != self.vit_dim
            else nn.Identity()
        )

        # Classifier head (named 'cls_head' to match NB08 checkpoint keys)
        self.cls_head = nn.Linear(embed_dim, num_classes, bias=False)
        if isinstance(self.proj, nn.Linear):
            nn.init.kaiming_normal_(self.proj.weight, mode="fan_out")
        nn.init.normal_(self.cls_head.weight, std=0.001)

        # JPM branch (training only)
        if self.jpm:
            self.bn_jpm = nn.BatchNorm1d(self.vit_dim)
            self.bn_jpm.bias.requires_grad_(False)
            self.jpm_cls = nn.Linear(self.vit_dim, num_classes, bias=False)
            nn.init.normal_(self.jpm_cls.weight, std=0.001)

    def forward(self, x: torch.Tensor, cam_ids: torch.Tensor | None = None):
        B = x.shape[0]
        rot_pos_embed = None

        # 1. Patch embedding
        x = self.vit.patch_embed(x)

        # 2. CLS token + positional embedding + pos_drop (use timm's method)
        if hasattr(self.vit, "_pos_embed"):
            result = self.vit._pos_embed(x)
            if isinstance(result, tuple):
                x, rot_pos_embed = result
            else:
                x = result
        else:
            cls_tok = self.vit.cls_token.expand(B, -1, -1)
            x = torch.cat([cls_tok, x], dim=1) + self.vit.pos_embed
            if hasattr(self.vit, "pos_drop"):
                x = self.vit.pos_drop(x)

        # 3. SIE: camera embedding broadcast to ALL tokens (per TransReID paper)
        if self.sie_camera and cam_ids is not None:
            x = x + self.sie_embed[cam_ids]  # (B,1,D) broadcasts to (B,N+1,D)

        # 4. Patch drop (Identity for most models, but call if present)
        if hasattr(self.vit, "patch_drop"):
            x = self.vit.patch_drop(x)

        # 5. CRITICAL: Pre-normalization (CLIP uses LayerNorm here!)
        #    Standard ViTs have Identity here, so this is a no-op for them.
        #    Skipping this for CLIP ViTs completely destroys pretrained features.
        if hasattr(self.vit, "norm_pre"):
            x = self.vit.norm_pre(x)

        # 6. Transformer blocks + final norm
        for blk in self.vit.blocks:
            if rot_pos_embed is not None:
                x = blk(x, rope=rot_pos_embed)
            else:
                x = blk(x)
        x = self.vit.norm(x)

        # CLS token → global feature
        g_feat = x[:, 0]
        bn = self.bn(g_feat)
        proj = self.proj(bn)

        if self.training:
            cls = self.cls_head(proj)
            if self.jpm:
                patches = x[:, 1:]
                idx = torch.randperm(patches.size(1), device=x.device)
                shuffled = patches[:, idx]
                mid = patches.size(1) // 2
                jpm_feat = (shuffled[:, :mid].mean(1) + shuffled[:, mid:].mean(1)) / 2
                jpm_cls = self.jpm_cls(self.bn_jpm(jpm_feat))
                return cls, proj, jpm_cls
            return cls, proj

        # Inference: L2-normalized embedding
        # If concat_patch is set, concatenate CLS with GeM-pooled patches
        proj_normed = F.normalize(proj, p=2, dim=1)
        if getattr(self, "_concat_patch", False):
            patches = x[:, 1:]  # (B, N, D) — patch tokens
            # GeM pooling: generalized mean with p=3 (more discriminative than avg)
            gem_p = getattr(self, "_gem_p", 3.0)
            patch_gem = (patches.clamp(min=1e-6) ** gem_p).mean(dim=1) ** (1.0 / gem_p)
            patch_normed = F.normalize(patch_gem, p=2, dim=1)
            return torch.cat([proj_normed, patch_normed], dim=1)
        return proj_normed


def build_transreid(
    num_classes: int = 1,
    num_cameras: int = 0,
    embed_dim: int = 768,
    vit_model: str = "vit_base_patch16_clip_224.openai",
    pretrained: bool = False,
    weights_path: str | None = None,
    img_size: tuple[int, int] | None = None,
) -> TransReID:
    """Build TransReID model and optionally load weights.

    Args:
        num_classes: Number of identity classes (1 for inference).
        num_cameras: Number of cameras for SIE (0 to disable).
        embed_dim: Output embedding dimension.
        vit_model: timm model name for the ViT backbone.
        pretrained: Load pretrained ViT weights from timm.
        weights_path: Path to trained TransReID checkpoint.
        img_size: (H, W) input image size.  When different from the timm
            default (224×224), timm creates the ViT with the correct
            patch grid and positional embedding length.

    Returns:
        TransReID model instance.
    """
    model = TransReID(
        num_classes=num_classes,
        num_cameras=num_cameras,
        embed_dim=embed_dim,
        vit_model=vit_model,
        pretrained=pretrained,
        sie_camera=num_cameras > 0,
        jpm=True,
        img_size=img_size,
    )

    if weights_path:
        state_dict = torch.load(weights_path, map_location="cpu", weights_only=False)
        if "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        elif "model" in state_dict:
            state_dict = state_dict["model"]
        elif "model_state_dict" in state_dict:
            state_dict = state_dict["model_state_dict"]

        # Strip module. prefix (from DataParallel)
        state_dict = {
            k.replace("module.", "", 1): v for k, v in state_dict.items()
        }

        # Remap 09p-style TransReIDViT keys to pipeline TransReID keys.
        remap_prefixes = {
            "bottleneck.": "bn.",
            "classifier.": "cls_head.",
        }
        remapped = {}
        for key, value in state_dict.items():
            new_key = key
            for old_prefix, new_prefix in remap_prefixes.items():
                if key.startswith(old_prefix):
                    new_key = new_prefix + key[len(old_prefix):]
                    break
            if new_key == "sie.camera_embed.weight":
                new_key = "sie_embed"
                value = value.unsqueeze(1)
            remapped[new_key] = value
        state_dict = remapped

        # ------------------------------------------------------------------
        # Handle shape mismatches between checkpoint and model:
        #   - pos_embed: training resolution may differ from timm default
        #   - sie_embed: checkpoint may have fewer cameras than deployment
        #   - cls_head / jpm_cls: num_classes differs at inference
        # Strategy: drop mismatched keys and let `strict=False` handle them.
        # For sie_embed specifically, we zero-pad if the checkpoint has fewer
        # cameras (safe: extra cameras just get zero SIE bias).
        # ------------------------------------------------------------------
        model_sd = model.state_dict()
        keys_to_drop = []
        for key in list(state_dict.keys()):
            if key not in model_sd:
                continue
            if state_dict[key].shape != model_sd[key].shape:
                if key == "sie_embed" and state_dict[key].shape[0] < model_sd[key].shape[0]:
                    # Zero-pad SIE: copy trained cameras, leave extras as zero
                    ckpt_cams = state_dict[key].shape[0]
                    padded = torch.zeros_like(model_sd[key])
                    padded[:ckpt_cams] = state_dict[key]
                    state_dict[key] = padded
                    logger.info(
                        f"SIE embed: padded {ckpt_cams} → "
                        f"{model_sd[key].shape[0]} cameras"
                    )
                elif key == "vit.pos_embed":
                    ckpt_pe = state_dict[key]  # (1, ckpt_tokens, D)
                    model_pe = model_sd[key]   # (1, model_tokens, D)
                    ckpt_tokens = ckpt_pe.shape[1]
                    model_tokens = model_pe.shape[1]
                    if ckpt_tokens == model_tokens:
                        pass  # same size, load directly
                    elif model_tokens > ckpt_tokens:
                        # Model requests larger grid (e.g. 384×384) than checkpoint
                        # (e.g. 256×256). Bicubic-interpolate grid embeddings.
                        cls_token = ckpt_pe[:, :1, :]  # (1, 1, D)
                        grid_pe = ckpt_pe[:, 1:, :]    # (1, N_ckpt, D)
                        ckpt_grid = int(grid_pe.shape[1] ** 0.5)
                        model_grid = int((model_tokens - 1) ** 0.5)
                        grid_pe = grid_pe.reshape(1, ckpt_grid, ckpt_grid, -1).permute(0, 3, 1, 2)
                        grid_pe = F.interpolate(
                            grid_pe.float(), size=(model_grid, model_grid),
                            mode="bicubic", align_corners=False,
                        ).to(ckpt_pe.dtype)
                        grid_pe = grid_pe.permute(0, 2, 3, 1).reshape(1, model_grid * model_grid, -1)
                        state_dict[key] = torch.cat([cls_token, grid_pe], dim=1)
                        logger.info(
                            f"Interpolated vit.pos_embed: {ckpt_grid}×{ckpt_grid} → "
                            f"{model_grid}×{model_grid} grid (bicubic)"
                        )
                    else:
                        # Checkpoint has more tokens — resize model to match
                        logger.info(
                            f"Resizing vit.pos_embed: model {model_sd[key].shape} → "
                            f"checkpoint {state_dict[key].shape}"
                        )
                        model.vit.pos_embed = nn.Parameter(
                            torch.zeros_like(state_dict[key]), requires_grad=False,
                        )
                else:
                    # Shape mismatch on classifier heads etc. — drop
                    logger.debug(
                        f"Dropping key {key}: ckpt {state_dict[key].shape} "
                        f"vs model {model_sd[key].shape}"
                    )
                    keys_to_drop.append(key)
        for key in keys_to_drop:
            del state_dict[key]

        # Load with relaxed strictness (num_classes may differ)
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        if missing:
            # Filter out classifier/cls_head which are expected to mismatch
            critical_missing = [
                k for k in missing
                if not any(skip in k for skip in ("cls_head", "classifier", "jpm_cls"))
            ]
            if critical_missing:
                logger.warning(f"TransReID critical missing keys: {critical_missing}")
            else:
                logger.debug(f"TransReID missing keys (non-critical): {len(missing)}")
        if unexpected:
            logger.debug(f"TransReID unexpected keys: {len(unexpected)}")
        logger.info(f"Loaded TransReID weights from {weights_path}")

    return model

## Inlined model: CLIP-SENet architecture (verbatim from `paper_veri776_A5alpha.ipynb` cell 24)

In [ ]:
"""CLIP-SENet architecture for vehicle re-identification.

Milestone M1 covers only the model port and local forward-pass smoke tests.
Training losses, camera/viewpoint embeddings, and Kaggle integration are
intentionally deferred.
"""

from __future__ import annotations

from dataclasses import dataclass

import logging
import torch
import torch.nn as nn
import torch.nn.functional as F

logger = logging.getLogger(__name__)
if not logger.handlers:
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(_h)
    logger.setLevel(logging.INFO)


@dataclass(frozen=True)
class LoadedBackboneInfo:
    """Describes the backbone variant that was successfully loaded."""

    family: str
    model_name: str
    pretrained_tag: str | None = None


class AFEMBlock(nn.Module):
    """Adaptive Fine-grained Enhancement Module.

    This implements the paper's ambiguous Eq. (4) using the `(G + 1)`
    interpretation: `G` grouped weighted residual chunks plus one identity
    residual path. Set `residual_mode="sum_only"` to drop the identity term and
    return only the weighted grouped sum.
    """

    def __init__(
        self,
        in_dim: int = 2048,
        out_dim: int = 2048,
        num_groups: int = 32,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        if out_dim % num_groups != 0:
            raise ValueError(
                f"AFEM out_dim={out_dim} must be divisible by num_groups={num_groups}"
            )
        if residual_mode not in {"grouped_identity", "sum_only"}:
            raise ValueError(
                "residual_mode must be 'grouped_identity' or 'sum_only'"
            )

        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_groups = num_groups
        self.group_dim = out_dim // num_groups
        self.residual_mode = residual_mode

        self.shared = nn.Sequential(
            nn.Linear(in_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
        )
        self.group_weights = nn.Parameter(torch.randn(num_groups, self.group_dim))

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        linear = self.shared[0]
        nn.init.kaiming_normal_(linear.weight, mode="fan_out")
        bn = self.shared[1]
        nn.init.ones_(bn.weight)
        nn.init.zeros_(bn.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.shared(x)
        grouped = h.view(h.shape[0], self.num_groups, self.group_dim)
        weighted = grouped * self.group_weights.unsqueeze(0)
        enhanced = weighted.reshape(h.shape[0], self.out_dim)
        if self.residual_mode == "sum_only":
            return enhanced
        return h + enhanced


class _ResNetFeatureWrapper(nn.Module):
    """Wrap torchvision-style ResNet backbones to expose pooled 2048-d features."""

    def __init__(self, model: nn.Module):
        super().__init__()
        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return torch.flatten(x, 1)


class ResNet101IBNBranch(nn.Module):
    """Appearance branch backed by real ResNet101 IBN-a with deterministic fallbacks."""

    _IBN_MODEL = "resnet101_ibn_a"
    _FALLBACK_MODEL = "resnet101"

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.output_dim = 2048
        self.backbone: nn.Module
        self.loaded_backbone: LoadedBackboneInfo

        for loader in (
            self._load_pretrainedmodels_ibn,
            self._load_torch_hub_ibn,
            self._load_timm_ibn,
            self._load_timm_plain,
        ):
            loaded = loader(pretrained=pretrained)
            if loaded is None:
                continue
            self.backbone, self.loaded_backbone = loaded
            logger.info(
                "Appearance branch loaded via '%s' model='%s' pretrained_tag='%s'",
                self.loaded_backbone.family,
                self.loaded_backbone.model_name,
                self.loaded_backbone.pretrained_tag,
            )
            return

        raise ImportError(
            "Unable to load appearance backbone via pretrainedmodels, torch.hub, or timm"
        )

    def _load_pretrainedmodels_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import pretrainedmodels
        except ImportError:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' is unavailable; trying torch.hub"
            )
            return None

        constructor = getattr(pretrainedmodels, self._IBN_MODEL, None)
        if constructor is None:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' has no '%s' entry; trying torch.hub",
                self._IBN_MODEL,
            )
            return None

        pretrained_tag = "imagenet" if pretrained else None
        try:
            raw_model = constructor(pretrained=pretrained_tag)
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "last_linear"):
            raw_model.last_linear = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="pretrainedmodels",
            model_name=self._IBN_MODEL,
            pretrained_tag=pretrained_tag or "random_init",
        )

    def _load_torch_hub_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            raw_model = torch.hub.load(
                "XingangPan/IBN-Net",
                self._IBN_MODEL,
                pretrained=pretrained,
                trust_repo=True,
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'torch.hub' failed for '{}': {}",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "fc"):
            raw_model.fc = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="torch.hub",
            model_name=self._IBN_MODEL,
            pretrained_tag="official_pretrained" if pretrained else "random_init",
        )

    def _load_timm_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        available = set(timm.list_models())
        if self._IBN_MODEL not in available:
            logger.warning(
                "Appearance branch loader 'timm' has no '%s' entry; trying plain '%s'",
                self._IBN_MODEL,
                self._FALLBACK_MODEL,
            )
            return None

        try:
            backbone = timm.create_model(
                self._IBN_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._IBN_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def _load_timm_plain(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        try:
            backbone = timm.create_model(
                self._FALLBACK_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for plain '%s': %s",
                self._FALLBACK_MODEL,
                exc,
            )
            return None

        logger.warning(
            "Appearance branch fell back to plain '%s' because no IBN-a loader succeeded",
            self._FALLBACK_MODEL,
        )
        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._FALLBACK_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x)
        if out.ndim != 2:
            raise RuntimeError(
                f"Appearance branch expected pooled 2D output, got shape {tuple(out.shape)}"
            )
        return out


class TinyCLIPImageBranch(nn.Module):
    """Semantic branch that loads TinyCLIP with a deterministic fallback chain."""

    _OPEN_CLIP_CHAIN = (
        {
            "model_name": "hf-hub:wkcn/TinyCLIP-ViT-45M-32-Text-21M-LAION400M",
            "pretrained_tag": None,
        },
        {
            "model_name": "TinyCLIP-ViT-40M-32-Text-19M",
            "pretrained_tag": "laion400m_e32",
        },
    )
    _TIMM_TINYCLIP_CHAIN = (
        "vit_medium_patch32_clip_224.tinyclip_laion400m",
    )
    _LAST_RESORT_OPEN_CLIP = ("ViT-B-32", "openai")

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.provider = ""
        self.model = None
        self.loaded_backbone: LoadedBackboneInfo | None = None
        last_error = self._try_load_open_clip(pretrained=pretrained)
        if self.model is None:
            last_error = self._try_load_timm_tinyclip(pretrained=pretrained) or last_error
        if self.model is None:
            last_error = self._try_load_open_clip_last_resort(pretrained=pretrained) or last_error

        if self.model is None or self.loaded_backbone is None:
            raise RuntimeError(
                "Unable to load any TinyCLIP/OpenCLIP visual backbone"
            ) from last_error

        self.image_size = self._infer_image_size(self.model)

    def _try_load_open_clip(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for candidate in self._OPEN_CLIP_CHAIN:
            model_name = candidate["model_name"]
            pretrained_tag = candidate["pretrained_tag"]
            try:
                if pretrained_tag is None:
                    model, _, _ = open_clip.create_model_and_transforms(model_name)
                else:
                    model, _, _ = open_clip.create_model_and_transforms(
                        model_name,
                        pretrained=pretrained_tag if pretrained else None,
                    )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP load failed for model='%s' pretrained='%s': %s",
                    model_name,
                    pretrained_tag or "hf-hub-default",
                    exc,
                )
                continue

            self.model = model
            self.provider = "open_clip"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag=pretrained_tag if pretrained else "random_init",
            )
            self.output_dim = self._infer_open_clip_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
                model_name,
                pretrained_tag if pretrained_tag is not None and pretrained else "hf-hub-default",
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_timm_tinyclip(self, pretrained: bool) -> Exception | None:
        try:
            import timm
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for model_name in self._TIMM_TINYCLIP_CHAIN:
            try:
                model = timm.create_model(
                    model_name,
                    pretrained=pretrained,
                    num_classes=0,
                )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP-equivalent timm load failed for model='%s': %s",
                    model_name,
                    exc,
                )
                continue

            self.model = model
            self.provider = "timm"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag="timm_pretrained" if pretrained else "random_init",
            )
            self.output_dim = self._infer_timm_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' via timm output_dim=%s",
                model_name,
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_open_clip_last_resort(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        model_name, pretrained_tag = self._LAST_RESORT_OPEN_CLIP
        try:
            model, _, _ = open_clip.create_model_and_transforms(
                model_name,
                pretrained=pretrained_tag if pretrained else None,
            )
        except Exception as exc:  # noqa: BLE001 - explicit last resort context
            logger.warning(
                "OpenCLIP last resort load failed for model='%s' pretrained='%s': %s",
                model_name,
                pretrained_tag,
                exc,
            )
            return exc

        self.model = model
        self.provider = "open_clip"
        self.loaded_backbone = LoadedBackboneInfo(
            family="semantic",
            model_name=model_name,
            pretrained_tag=pretrained_tag if pretrained else "random_init",
        )
        self.output_dim = self._infer_open_clip_output_dim(model)
        logger.info(
            "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
            model_name,
            pretrained_tag if pretrained else "random_init",
            self.output_dim,
        )
        return None

    @staticmethod
    def _infer_open_clip_output_dim(model: nn.Module) -> int:
        visual = getattr(model, "visual", None)
        output_dim = getattr(visual, "output_dim", None)
        if isinstance(output_dim, int):
            return output_dim

        visual_proj = getattr(model, "visual_projection", None)
        if isinstance(visual_proj, torch.Tensor) and visual_proj.ndim == 2:
            return int(visual_proj.shape[-1])

        visual_proj = getattr(visual, "proj", None)
        if isinstance(visual_proj, torch.Tensor):
            if visual_proj.ndim == 1:
                return int(visual_proj.shape[0])
            if visual_proj.ndim == 2:
                return int(visual_proj.shape[-1])

        raise RuntimeError("Could not infer TinyCLIP visual output dimension")

    @staticmethod
    def _infer_timm_output_dim(model: nn.Module) -> int:
        output_dim = getattr(model, "num_features", None)
        if isinstance(output_dim, int):
            return output_dim
        raise RuntimeError("Could not infer timm TinyCLIP visual output dimension")

    @staticmethod
    def _infer_image_size(model: nn.Module) -> tuple[int, int]:
        pretrained_cfg = getattr(model, "pretrained_cfg", None)
        if isinstance(pretrained_cfg, dict):
            input_size = pretrained_cfg.get("input_size")
            if isinstance(input_size, tuple) and len(input_size) == 3:
                return (int(input_size[-2]), int(input_size[-1]))

        visual = getattr(model, "visual", None)
        image_size = getattr(visual, "image_size", None)
        if isinstance(image_size, int):
            return (image_size, image_size)
        if isinstance(image_size, tuple) and len(image_size) == 2:
            return (int(image_size[0]), int(image_size[1]))
        return (224, 224)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if tuple(x.shape[-2:]) != self.image_size:
            x = F.interpolate(
                x,
                size=self.image_size,
                mode="bilinear",
                align_corners=False,
            )
        if self.provider == "open_clip":
            features = self.model.encode_image(x, normalize=False)
        else:
            features = self.model(x)
        if features.ndim != 2:
            raise RuntimeError(
                f"TinyCLIP branch expected 2D image features, got shape {tuple(features.shape)}"
            )
        return features


class CLIPSENet(nn.Module):
    """CLIP-SENet with a CNN appearance branch and a CLIP semantic branch."""

    def __init__(
        self,
        num_classes: int,
        embed_dim: int = 2048,
        afem_groups: int = 32,
        feat_dim_appearance: int = 2048,
        feat_dim_semantic: int = 512,
        dropout: float = 0.0,
        appearance_pretrained: bool = True,
        semantic_pretrained: bool = True,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim

        self.appearance_branch = ResNet101IBNBranch(pretrained=appearance_pretrained)
        self.semantic_branch = TinyCLIPImageBranch(pretrained=semantic_pretrained)

        detected_app_dim = self.appearance_branch.output_dim
        detected_sem_dim = self.semantic_branch.output_dim
        if feat_dim_appearance != detected_app_dim:
            logger.warning(
                "Requested feat_dim_appearance=%s but backbone reports %s. Using detected dim.",
                feat_dim_appearance,
                detected_app_dim,
            )
        if feat_dim_semantic != detected_sem_dim:
            logger.warning(
                "Requested feat_dim_semantic=%s but backbone reports %s. Using detected dim.",
                feat_dim_semantic,
                detected_sem_dim,
            )

        self.feat_dim_appearance = detected_app_dim
        self.feat_dim_semantic = detected_sem_dim
        self.fusion_fc = nn.Linear(
            self.feat_dim_appearance + self.feat_dim_semantic,
            embed_dim,
            bias=False,
        )
        self.afem = AFEMBlock(
            in_dim=embed_dim,
            out_dim=embed_dim,
            num_groups=afem_groups,
            residual_mode=residual_mode,
        )
        self.bnneck = nn.BatchNorm1d(embed_dim)
        self.bnneck.bias.requires_grad_(False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.classifier = nn.Linear(embed_dim, num_classes, bias=False)

        nn.init.kaiming_normal_(self.fusion_fc.weight, mode="fan_out")
        nn.init.normal_(self.classifier.weight, std=0.001)

        self.loaded_resnext_model = self.appearance_branch.loaded_backbone.model_name
        self.loaded_tinyclip_model = self.semantic_branch.loaded_backbone.model_name

    def forward(self, x: torch.Tensor):
        f_app = self.appearance_branch(x)
        f_sem = self.semantic_branch(x)
        t_u = self.fusion_fc(torch.cat([f_app, f_sem], dim=1))
        t_s_prime = self.afem(t_u)
        t = t_u + t_s_prime
        t_bn = self.bnneck(t)
        t_bn_normalized = F.normalize(t_bn, p=2, dim=1)

        if self.training:
            logits = self.classifier(self.dropout(t_bn))
            return t_bn_normalized, logits

        return t_bn_normalized


def build_clip_senet(num_classes: int, **kwargs) -> CLIPSENet:
    """Build a CLIP-SENet model for M1 architecture validation."""

    return CLIPSENet(num_classes=num_classes, **kwargs)

## Eval / AQE / k-reciprocal rerank helpers (verbatim from `paper_veri776_A5alpha.ipynb` cell 25)

In [ ]:
def l2_normalize(features):
    features = features.astype(np.float32, copy=False)
    return features / (np.linalg.norm(features, axis=1, keepdims=True) + 1e-12)


def compute_distance_from_similarity(similarity):
    return (1.0 - similarity).astype(np.float32, copy=False)


def eval_market1501(distmat, q_pids, g_pids, q_camids, g_camids, max_rank=50):
    num_q, num_g = distmat.shape
    if num_g < max_rank:
        max_rank = num_g
    indices = np.argsort(distmat, axis=1)
    matches = (g_pids[indices] == q_pids[:, np.newaxis]).astype(np.int32)
    all_cmc = []
    all_ap = []
    num_valid = 0
    for q_idx in range(num_q):
        q_pid = q_pids[q_idx]
        q_camid = q_camids[q_idx]
        order = indices[q_idx]
        remove = (g_pids[order] == q_pid) & (g_camids[order] == q_camid)
        keep = ~remove
        if not np.any(matches[q_idx][keep]):
            continue
        raw_cmc = matches[q_idx][keep]
        num_valid += 1
        cmc = raw_cmc.cumsum()
        cmc[cmc > 1] = 1
        all_cmc.append(cmc[:max_rank])
        num_rel = raw_cmc.sum()
        tmp_cmc = raw_cmc.cumsum()
        precision = tmp_cmc / (np.arange(len(tmp_cmc)) + 1.0)
        ap = (precision * raw_cmc.astype(bool)).sum() / num_rel if num_rel > 0 else 0.0
        all_ap.append(ap)
    if num_valid == 0:
        raise RuntimeError("No valid query found during VeRi evaluation")
    cmc = np.asarray(all_cmc, dtype=np.float32).mean(axis=0)
    return float(np.mean(all_ap)), cmc


def to_metric_dict(mAP, cmc):
    ranks = list(cmc)
    return {
        "mAP": float(mAP),
        "R1": float(ranks[min(0, len(ranks) - 1)]),
        "R5": float(ranks[min(4, len(ranks) - 1)]),
        "R10": float(ranks[min(9, len(ranks) - 1)]),
    }


def metric_sort_key(metrics):
    return (metrics["mAP"], metrics["R1"], metrics["R5"], metrics["R10"])


def print_metrics(label, metrics, duration_sec=None):
    suffix = "" if duration_sec is None else f" ({duration_sec:.1f}s)"
    print(
        f"{label}: mAP={metrics['mAP'] * 100:.4f}% "
        f"R1={metrics['R1'] * 100:.4f}% "
        f"R5={metrics['R5'] * 100:.2f}% "
        f"R10={metrics['R10'] * 100:.2f}%{suffix}"
    )


def average_query_expansion(features, k, iterations=1):
    current = l2_normalize(features.copy())
    if k <= 1:
        return current
    for _ in range(iterations):
        sim = current @ current.T
        topk = min(k, sim.shape[1])
        kth = max(topk - 1, 0)
        topk_idx = np.argpartition(-sim, kth=kth, axis=1)[:, :topk]
        expanded = np.zeros_like(current, dtype=np.float32)
        for index in range(current.shape[0]):
            expanded[index] = current[topk_idx[index]].mean(axis=0)
        current = l2_normalize(expanded)
    return current


@torch.no_grad()
def build_rerank_state_from_similarity(similarity, max_k1):
    sim_tensor = torch.as_tensor(similarity, dtype=torch.float32, device=DEVICE)
    original_dist = (2.0 - 2.0 * sim_tensor).clamp_min_(0).cpu().numpy().astype(np.float32)
    initial_rank = torch.topk(
        sim_tensor,
        k=min(max_k1 + 1, sim_tensor.shape[1]),
        dim=1,
        largest=True,
        sorted=True,
    ).indices.cpu().numpy().astype(np.int32)
    del sim_tensor
    if DEVICE.startswith("cuda"):
        torch.cuda.empty_cache()
    return original_dist, initial_rank


def compute_reranking_torch(original_dist, initial_rank, query_num, k1=80, k2=15, lambda_value=0.2):
    all_num = original_dist.shape[0]
    V = np.zeros((all_num, all_num), dtype=np.float16)
    half_k1 = int(np.round(k1 / 2.0))
    for index in range(all_num):
        forward = initial_rank[index, :k1 + 1]
        backward = initial_rank[forward, :k1 + 1]
        reciprocal = forward[np.any(backward == index, axis=1)]
        reciprocal_expansion = reciprocal.copy()
        for candidate in reciprocal:
            candidate_forward = initial_rank[candidate, :half_k1 + 1]
            candidate_backward = initial_rank[candidate_forward, :half_k1 + 1]
            candidate_reciprocal = candidate_forward[np.any(candidate_backward == candidate, axis=1)]
            if candidate_reciprocal.size == 0:
                continue
            overlap = np.intersect1d(candidate_reciprocal, reciprocal)
            if overlap.size > (2.0 / 3.0) * candidate_reciprocal.size:
                reciprocal_expansion = np.concatenate((reciprocal_expansion, candidate_reciprocal))
        reciprocal_expansion = np.unique(reciprocal_expansion)
        weights = np.exp(-original_dist[index, reciprocal_expansion]).astype(np.float32)
        V[index, reciprocal_expansion] = (weights / (weights.sum() + 1e-12)).astype(np.float16)
    if k2 > 1:
        V_qe = np.zeros_like(V, dtype=np.float16)
        for index in range(all_num):
            V_qe[index] = V[initial_rank[index, :k2]].mean(axis=0)
        V = V_qe
    inv_index = [np.flatnonzero(V[:, column]) for column in range(all_num)]
    jaccard_dist = np.zeros((query_num, all_num), dtype=np.float32)
    for index in range(query_num):
        temp_min = np.zeros(all_num, dtype=np.float32)
        non_zero = np.flatnonzero(V[index])
        for nz in non_zero:
            related = inv_index[nz]
            temp_min[related] += np.minimum(np.float32(V[index, nz]), V[related, nz].astype(np.float32))
        jaccard_dist[index] = 1.0 - temp_min / (2.0 - temp_min)
    final_dist = jaccard_dist * (1.0 - lambda_value) + original_dist[:query_num] * lambda_value
    return final_dist[:, query_num:]


def evaluate_distmat(label, distmat, row):
    started = time.time()
    mAP, cmc = eval_market1501(distmat, q_pids, g_pids, q_camids, g_camids)
    metrics = to_metric_dict(mAP, cmc)
    metrics["duration_sec"] = float(time.time() - started)
    record = {**row, "label": label, "metrics": metrics}
    print_metrics(label, metrics, metrics["duration_sec"])
    return record

## VeRi-776 discovery, parse, transforms (adapted from `14t_veri_fusion.ipynb` cell 3)

In [ ]:
# === VeRi-776 discovery + parse + transforms ===
CAMERA_PATTERN = re.compile(r"^(?P<pid>-?\d+)_c(?P<camid>\d+)")
NUM_VERI_CAMERAS = 20

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

TRANSREID_IMG_SIZE = (224, 224)
CLIPSENET_IMG_SIZE = (320, 320)
TRANSREID_BATCH_SIZE = 64
CLIPSENET_BATCH_SIZE = 32
NUM_WORKERS = 4
CONCAT_PATCH_GEM_P = 3.0
VIT_MODEL = "vit_base_patch16_clip_224.openai"  # hardcoded (spec R7) — no dynamic find_clip_vit_base


def discover_veri_root() -> Path:
    for candidate in Path("/kaggle/input").rglob("*"):
        if candidate.is_dir() and (candidate / "image_query").is_dir() and (candidate / "image_test").is_dir():
            return candidate
    raise FileNotFoundError("VeRi-776 root not found under /kaggle/input")


def parse_veri_record(img_path: Path) -> dict:
    match = CAMERA_PATTERN.match(img_path.stem)
    if match is None:
        raise RuntimeError(f"Unexpected VeRi filename: {img_path.name}")
    parsed_camid = int(match.group("camid"))
    if not 1 <= parsed_camid <= NUM_VERI_CAMERAS:
        raise RuntimeError(f"Camera ID out of range for {img_path.name}: c{parsed_camid:03d}")
    return {
        "path": str(img_path),
        "pid": int(match.group("pid")),
        "camid": parsed_camid - 1,
        "sie_index": parsed_camid - 1,
    }


def parse_split(split_dir: Path):
    items = []
    pid_set = set()
    for img_path in sorted(split_dir.glob("*.jpg")):
        record = parse_veri_record(img_path)
        if record["pid"] == -1:
            continue
        pid_set.add(record["pid"])
        items.append(record)
    return items, len(pid_set)


class VeRiTensorDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        item = self.items[index]
        image = Image.open(item["path"]).convert("RGB")
        return self.transform(image), int(item["pid"]), int(item["camid"]), int(item["sie_index"]), item["path"]


def build_transform(size, mean, std):
    return T.Compose([
        T.Resize(size, interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


def build_tensor_loader(items, size, mean, std, batch_size):
    return DataLoader(
        VeRiTensorDataset(items, build_transform(size, mean, std)),
        batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
    )


VERI_ROOT = discover_veri_root()
query_items, query_ids = parse_split(VERI_ROOT / "image_query")
gallery_items, gallery_ids = parse_split(VERI_ROOT / "image_test")
print("VERI_ROOT:", VERI_ROOT)
print(f"Query:   {len(query_items):,} images, {query_ids} IDs")
print(f"Gallery: {len(gallery_items):,} images, {gallery_ids} IDs")

# Spec R4: assert dataset counts (pin abhyudaya12/veri-... and fail loud on drift)
assert len(query_items) == 1678, f"Expected query=1678, got {len(query_items)}"
assert len(gallery_items) == 11579, f"Expected gallery=11579, got {len(gallery_items)}"
print("Dataset counts OK (query=1678, gallery=11579).")

## Load the two FROZEN canonical checkpoints, log SHA-256, assert backbone provenance (spec R5/R6)

In [ ]:
# === Checkpoint loading: canonical names, SHA-256, MANIFEST assert, backbone provenance ===
def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def _find_ckpt(name):
    p = CANON_WEIGHTS_DIR / name
    if p.is_file():
        return p
    hits = list(Path('/kaggle/input').rglob(name))
    assert hits, f'{name} not found anywhere under /kaggle/input'
    print(f'Resolved {name} -> {hits[0]}')
    return hits[0]
STREAM1_CKPT = _find_ckpt(STREAM1_CKPT_NAME)
STREAM2_CKPT = _find_ckpt(STREAM2_CKPT_NAME)
assert STREAM1_CKPT.is_file(), f"Missing Stream-1 checkpoint: {STREAM1_CKPT}"
assert STREAM2_CKPT.is_file(), f"Missing Stream-2 checkpoint: {STREAM2_CKPT}"

STREAM1_SHA = sha256_of(STREAM1_CKPT)
STREAM2_SHA = sha256_of(STREAM2_CKPT)
print(f"Stream-1 SHA-256: {STREAM1_SHA}  ({STREAM1_CKPT})")
print(f"Stream-2 SHA-256: {STREAM2_SHA}  ({STREAM2_CKPT})")

# If a MANIFEST.json ships with the dataset, assert SHA matches; else just log (manifest added at publish time).
MANIFEST_PATH = CANON_WEIGHTS_DIR / "MANIFEST.json"
MANIFEST = None
if MANIFEST_PATH.is_file():
    MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    def _manifest_sha(name):
        # accept either {name: sha} or {name: {sha256: ...}} shapes
        entry = MANIFEST.get(name)
        if isinstance(entry, dict):
            return entry.get("sha256") or entry.get("sha")
        return entry
    exp1 = _manifest_sha(STREAM1_CKPT_NAME)
    exp2 = _manifest_sha(STREAM2_CKPT_NAME)
    if exp1:
        assert exp1 == STREAM1_SHA, f"Stream-1 SHA mismatch vs MANIFEST: {exp1} != {STREAM1_SHA}"
    if exp2:
        assert exp2 == STREAM2_SHA, f"Stream-2 SHA mismatch vs MANIFEST: {exp2} != {STREAM2_SHA}"
    print("MANIFEST.json present — SHA-256 assertions passed.")
else:
    print("No MANIFEST.json in dataset dir — logging SHA-256 only (manifest added when dataset is published).")


# --- Stream 1: TransReID via inlined build_transreid (NO repo clone, NO src. import) ---
def load_stream1_transreid(weights_path):
    m = build_transreid(
        num_classes=575, num_cameras=NUM_VERI_CAMERAS, embed_dim=768,
        vit_model=VIT_MODEL, pretrained=False,
        weights_path=str(weights_path), img_size=TRANSREID_IMG_SIZE,
    )
    m._concat_patch = True   # 1536-D concat_patch inference path
    m._gem_p = CONCAT_PATCH_GEM_P
    return m.to(DEVICE).eval()


# --- Stream 2: CLIP-SENet, bare state_dict, strict load, assert backbone provenance ---
def load_stream2_clipsenet(weights_path):
    payload = torch_load(weights_path)
    if isinstance(payload, dict) and "model_state" in payload:
        state_dict = payload["model_state"]
        kind = "payload:model_state"
    elif isinstance(payload, dict) and "model" in payload and isinstance(payload["model"], dict):
        state_dict = payload["model"]
        kind = "payload:model"
    elif isinstance(payload, dict) and payload and all(hasattr(v, "shape") for v in payload.values()):
        state_dict = payload
        kind = "state_dict"
    else:
        raise TypeError(f"Unsupported CLIP-SENet checkpoint format: {type(payload).__name__}")
    classifier_weight = state_dict.get("classifier.weight")
    inferred_num_classes = int(classifier_weight.shape[0]) if classifier_weight is not None else 575
    m = build_clip_senet(num_classes=inferred_num_classes).to(DEVICE)
    # Spec R5: fail hard if the appearance/semantic backbone fell back to a different model.
    app_model = m.appearance_branch.loaded_backbone.model_name
    sem_info = m.semantic_branch.loaded_backbone
    assert app_model == "resnet101_ibn_a", (
        f"Appearance backbone fell back to {app_model!r}; expected resnet101_ibn_a (checkpoint mismatch)"
    )
    assert "tinyclip" in sem_info.model_name.lower(), (
        f"Semantic backbone is {sem_info.model_name!r}; expected a TinyCLIP variant (canonical timm vit_medium_patch32) (fallback would mismatch the checkpoint)"
    )
    missing, unexpected = m.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        print("Missing keys:", missing)
        print("Unexpected keys:", unexpected)
        raise RuntimeError("CLIP-SENet checkpoint load was not strict")
    m.eval()
    return m, {
        "checkpoint_kind": kind,
        "num_classes": inferred_num_classes,
        "appearance_model": app_model,
        "semantic_model": sem_info.model_name,
        "semantic_dim": int(m.feat_dim_semantic),
    }


stream1_model = load_stream1_transreid(STREAM1_CKPT)
stream2_model, stream2_info = load_stream2_clipsenet(STREAM2_CKPT)
print("Stream-2 load info:", json.dumps(stream2_info, indent=2))
print("Backbone provenance OK (resnet101_ibn_a + TinyCLIP timm vit_medium_patch32).")

## Feature extraction (Stream-1 1536-D flip-TTA, Stream-2 2048-D) + fp16 feature dump (from `14t` cells 5-6 / A5alpha cell 27)

In [ ]:
# === Feature extraction (verbatim helpers from 14t cell 5) ===
@torch.no_grad()
def extract_transreid_features(model, items, mode: str):
    loader = build_tensor_loader(items, TRANSREID_IMG_SIZE, CLIP_MEAN, CLIP_STD, TRANSREID_BATCH_SIZE)
    features = []
    pids = []
    camids = []
    paths = []
    model.eval()
    model._concat_patch = mode == "concat_patch_flip"
    model._gem_p = CONCAT_PATCH_GEM_P
    started = time.time()
    for images, batch_pids, batch_camids, batch_sie, batch_paths in loader:
        images = images.to(DEVICE, non_blocking=True)
        batch_sie = batch_sie.to(DEVICE, non_blocking=True).long()
        forward_views = []
        for view in (images, torch.flip(images, dims=[3])):
            output = model(view, cam_ids=batch_sie)
            if isinstance(output, (tuple, list)):
                output = output[-1]
            forward_views.append(F.normalize(output.float(), p=2, dim=1).cpu())
        batch_features = F.normalize(torch.stack(forward_views, dim=0).mean(dim=0), p=2, dim=1)
        features.append(batch_features.numpy().astype(np.float32))
        pids.append(batch_pids.numpy())
        camids.append(batch_camids.numpy())
        paths.extend(list(batch_paths))
    elapsed = time.time() - started
    merged = np.concatenate(features, axis=0)
    print(f"TransReID {mode}: {merged.shape} in {elapsed:.1f}s")
    model._concat_patch = False
    return merged, np.concatenate(pids), np.concatenate(camids), paths


@torch.no_grad()
def extract_clipsenet_features(model, items):
    loader = build_tensor_loader(items, CLIPSENET_IMG_SIZE, IMAGENET_MEAN, IMAGENET_STD, CLIPSENET_BATCH_SIZE)
    features = []
    pids = []
    camids = []
    paths = []
    started = time.time()
    model.eval()
    for images, batch_pids, batch_camids, _, batch_paths in loader:
        images = images.to(DEVICE, non_blocking=True)
        output = model(images)
        if isinstance(output, (tuple, list)):
            output = output[-1]
        output = F.normalize(output.float(), p=2, dim=1)
        features.append(output.cpu().numpy().astype(np.float32))
        pids.append(batch_pids.numpy())
        camids.append(batch_camids.numpy())
        paths.extend(list(batch_paths))
    elapsed = time.time() - started
    merged = np.concatenate(features, axis=0)
    print(f"CLIP-SENet: {merged.shape} in {elapsed:.1f}s")
    return merged, np.concatenate(pids), np.concatenate(camids), paths


q_tr_1536, q_pids, q_camids, query_paths = extract_transreid_features(stream1_model, query_items, "concat_patch_flip")
g_tr_1536, g_pids, g_camids, gallery_paths = extract_transreid_features(stream1_model, gallery_items, "concat_patch_flip")
q_tr_768, _, _, _ = extract_transreid_features(stream1_model, query_items, "single")
g_tr_768, _, _, _ = extract_transreid_features(stream1_model, gallery_items, "single")
q_cs, q_pids_cs, q_camids_cs, _ = extract_clipsenet_features(stream2_model, query_items)
g_cs, g_pids_cs, g_camids_cs, _ = extract_clipsenet_features(stream2_model, gallery_items)

assert np.array_equal(q_pids, q_pids_cs) and np.array_equal(g_pids, g_pids_cs)
assert np.array_equal(q_camids, q_camids_cs) and np.array_equal(g_camids, g_camids_cs)
assert q_tr_1536.shape[1] == 1536, f"Stream-1 dim {q_tr_1536.shape[1]} != 1536"
assert q_cs.shape[1] == 2048, f"Stream-2 dim {q_cs.shape[1]} != 2048"

# Provide aliases used by eval helpers (evaluate_distmat reads global q_pids/g_pids/q_camids/g_camids).
q_pids = q_pids.astype(np.int64); g_pids = g_pids.astype(np.int64)
q_camids = q_camids.astype(np.int64); g_camids = g_camids.astype(np.int64)

# --- Feature dump (fp16 .npy + index_map.json) for retrieval panels (like A5alpha cell 27) ---
OUT_DIR = WORKING / "features"
(OUT_DIR / "stream1").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "stream2").mkdir(parents=True, exist_ok=True)
np.save(OUT_DIR / "stream1" / "query.npy", q_tr_1536.astype(np.float16))
np.save(OUT_DIR / "stream1" / "gallery.npy", g_tr_1536.astype(np.float16))
np.save(OUT_DIR / "stream2" / "query.npy", q_cs.astype(np.float16))
np.save(OUT_DIR / "stream2" / "gallery.npy", g_cs.astype(np.float16))
index_map = {
    "query": [
        {"row": i, "image_path": qp, "vehicle_id": int(qid), "camera_id": int(qc), "split": "query"}
        for i, (qp, qid, qc) in enumerate(zip(query_paths, q_pids, q_camids))
    ],
    "gallery": [
        {"row": i, "image_path": gp, "vehicle_id": int(gid), "camera_id": int(gc), "split": "gallery"}
        for i, (gp, gid, gc) in enumerate(zip(gallery_paths, g_pids, g_camids))
    ],
    "stream1": {"dim": int(q_tr_1536.shape[1]), "tta": "concat_patch_flip", "checkpoint": str(STREAM1_CKPT), "sha256": STREAM1_SHA},
    "stream2": {"dim": int(q_cs.shape[1]), "tta": "none", "checkpoint": str(STREAM2_CKPT), "sha256": STREAM2_SHA},
}
with open(OUT_DIR / "index_map.json", "w", encoding="utf-8") as f:
    json.dump(index_map, f, indent=2)
print("Dumped fp16 features + index_map.json to", OUT_DIR)

# Free the models — all remaining work is numpy on the extracted features.
del stream1_model  # keep stream2_model alive for S2 flip-TTA extraction
gc.collect()
if DEVICE.startswith("cuda"):
    torch.cuda.empty_cache()

## Paper-table verification (exact params, NO sweep) from ORIGINAL deployed weights
Reproduces each row: S1 trajectory -> 89.97, S2 trajectory -> 91.54, Fusion -> 93.30.

In [ ]:
# === FUSION BOOST SWEEP (eval-only, frozen S1=vehicle_transreid + S2=v6) ===
# Levers: S1 dim {768 single-flip, 1536 concat-flip} x S2 {320 no-TTA, 320 flip-TTA} x w_cs grid.
@torch.no_grad()
def extract_cs_base_flip(items, size=320):
    loader = build_tensor_loader(items, (size, size), IMAGENET_MEAN, IMAGENET_STD, CLIPSENET_BATCH_SIZE)
    base, flip = [], []
    for images, *_ in loader:
        images = images.to(DEVICE, non_blocking=True)
        for tgt, v in ((base, images), (flip, torch.flip(images, dims=[3]))):
            o = stream2_model(v); o = o[-1] if isinstance(o, (tuple, list)) else o
            tgt.append(F.normalize(o.float(), p=2, dim=1).cpu().numpy().astype(np.float32))
    return np.concatenate(base, 0), np.concatenate(flip, 0)

print("extract S2 base+flip @320 ...")
q_cs_b, q_cs_f = extract_cs_base_flip(query_items); g_cs_b, g_cs_f = extract_cs_base_flip(gallery_items)
# --- v8 STANDALONE (compare to v6: base 82.34 / AQE k10 89.21 / +rerank 91.54) ---
def s2_standalone(qf, gf):
    base = to_metric_dict(*eval_market1501(compute_distance_from_similarity((l2_normalize(qf) @ l2_normalize(gf).T).astype(np.float32)), q_pids, g_pids, q_camids, g_camids))
    a = average_query_expansion(np.concatenate([l2_normalize(qf), l2_normalize(gf)], 0), k=10, iterations=1)
    aqe = to_metric_dict(*eval_market1501(compute_distance_from_similarity((a[:len(qf)] @ a[len(qf):].T).astype(np.float32)), q_pids, g_pids, q_camids, g_camids))
    od, ir = build_rerank_state_from_similarity((a @ a.T).astype(np.float32), max_k1=50)
    rr = to_metric_dict(*eval_market1501(compute_reranking_torch(od, ir, len(qf), k1=50, k2=10, lambda_value=0.1), q_pids, g_pids, q_camids, g_camids))
    return base, aqe, rr
_b, _a, _rr = s2_standalone(q_cs_b, g_cs_b)
print("=" * 60)
print(f"v8 STANDALONE: base mAP={_b['mAP']*100:.2f}/{_b['R1']*100:.2f}  AQE10 mAP={_a['mAP']*100:.2f}/{_a['R1']*100:.2f}  +rerank mAP={_rr['mAP']*100:.2f}/{_rr['R1']*100:.2f}")
print(f"  (v6 reference: 82.34 / 89.21 / 91.54)")
print("=" * 60)
def favg(b, f): return l2_normalize((l2_normalize(b) + l2_normalize(f)) / 2.0)

def fuse(q_tr, g_tr, q_cs, g_cs, w_cs):
    a_tr = average_query_expansion(np.concatenate([l2_normalize(q_tr), l2_normalize(g_tr)], 0), k=3, iterations=1)
    a_cs = average_query_expansion(np.concatenate([l2_normalize(q_cs), l2_normalize(g_cs)], 0), k=3, iterations=1)
    sim = ((1 - w_cs) * (a_tr @ a_tr.T) + w_cs * (a_cs @ a_cs.T)).astype(np.float32)
    od, ir = build_rerank_state_from_similarity(sim, max_k1=80)
    return to_metric_dict(*eval_market1501(compute_reranking_torch(od, ir, len(q_tr), k1=80, k2=15, lambda_value=0.2), q_pids, g_pids, q_camids, g_camids))

S1S = {"S1_768": (q_tr_768, g_tr_768), "S1_1536": (q_tr_1536, g_tr_1536)}
S2S = {"S2_noTTA": (q_cs_b, g_cs_b), "S2_flipTTA": (favg(q_cs_b, q_cs_f), favg(g_cs_b, g_cs_f))}
rows = []
for s1n, (qt, gt) in S1S.items():
    for s2n, (qc, gc) in S2S.items():
        for w in (0.60, 0.65, 0.70, 0.75):
            m = fuse(qt, gt, qc, gc, w)
            rows.append({"s1": s1n, "s2": s2n, "w_cs": w, "mAP": m["mAP"], "R1": m["R1"]})
            print(f"  {s1n:8} x {s2n:11} w_cs={w} -> mAP={m['mAP']*100:6.3f}  R1={m['R1']*100:6.3f}")
best = max(rows, key=lambda r: r["mAP"])
print("\n" + "=" * 70)
print(f"BEST: {best['s1']} x {best['s2']} w_cs={best['w_cs']} -> mAP={best['mAP']*100:.3f} R1={best['R1']*100:.3f}")
print("Reference: published/locked headline = 93.30 mAP")
print("=" * 70)
with open(WORKING / "eval_results.json", "w", encoding="utf-8") as f:
    json.dump({"fusion_boost": rows, "best": best,
               "weights": {"s1": STREAM1_CKPT_NAME, "s1_sha": STREAM1_SHA, "s2": STREAM2_CKPT_NAME, "s2_sha": STREAM2_SHA},
               "gpu": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")}, f, indent=2)
print("wrote eval_results.json")